[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Adding and Changing Columns


## What you will be able to do

Add columns to a table, change the values in chosen rows, and tell whether a change reached the table
you meant, including what `SettingWithCopyWarning` is telling you when it appears.

## The idea

### The problem

Most work on a table adds to it. A reading in Fahrenheit goes beside the one in Celsius, a column says
whether the hour was below freezing, and another holds each station's latitude, looked up from a
second table. Other work changes what is there, as when a sensor at Tromso read half a degree high for
a week and those readings need correcting before anything is averaged.

pandas has several ways to do each of these, and they look alike. Square brackets on the left of `=`,
the `assign` method and `loc` all put values into a column. They differ in which table ends up
changed. The usual symptom of a change aimed at the wrong table is a long warning about "a copy of a
slice", printed after a line that looked correct, and a table that may or may not have changed. A
correction that was supposed to fix the data can then miss it without an error.

### What assigning to a column is

> **Assigning to a column**, `df["name"] = values`, adds the column to `df`, or replaces it, in place.
> **`df.assign(name=values)`** returns a new table with the column and leaves `df` as it was.
> **`df.loc[rows, "name"] = value`** changes the chosen rows of one column of `df`, in a single step.

### Why it works that way

The values on the right are lined up with the table before they are stored. A single value goes into
every row. A list must have exactly one value for each row. A Series is lined up by its index, as
everything in pandas is, so a Series whose labels are not the table's row labels fills the column with
`NaN`.

A filtered table, such as `readings[readings["celsius"] > 15]`, is a new DataFrame with data of its
own, since a mask always makes a copy. Other selections, such as a single column, can share memory
with the table they came from, and pandas 2.2 does not say which one you have. So when you change a
column of any table cut from another, pandas cannot tell whether you meant to change the original too,
and it prints `SettingWithCopyWarning`. The warning means your change went into the table you changed
and may not have reached the original. `.copy()` says the table is yours to change, and `loc` changes
the original directly.

The worst case is two steps on one line, `readings[mask]["celsius"] = 0`. The first step makes a new
table, the second changes that table, and the new table is then thrown away. `readings` is untouched.
This is **chained assignment**, and `loc` with the mask and the column name does the same job in one
step.

pandas also has a mode called **Copy-on-Write**, in which a table taken from another always behaves as
a copy of its own. pandas 3.0 makes it the only mode, and pandas 2.2 can switch it on. This notebook
switches it on in the cells that show it, and off again afterward.

### Where this shows up

Every cleaning step before an analysis adds or corrects columns. The **Missing Data** notebook fills
gaps in a column, the **Aggregating** notebook adds each reading's distance from its station's mean,
and the **scikit-learn, Deep Dive** guide builds the columns a model learns from.

### What this notebook covers

- a column from arithmetic, from a condition and from another table
- `assign`, and how it leaves the original alone
- `loc` for changing chosen rows of one column
- `rename` and `drop`
- a filtered table, the warning, and `.copy()`
- Copy-on-Write, switched on to show what changes
- five errors, two of them silent

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import pandas as pd

readings = pd.DataFrame({"station": ["Oslo", "Oslo", "Bergen"], "celsius": [-1.5, 2.0, 4.5]})
readings["fahrenheit"] = readings["celsius"] * 9 / 5 + 32
readings.loc[readings["celsius"] < 0, "frost"] = True
print(readings)
```

```
  station  celsius  fahrenheit frost
0    Oslo     -1.5        29.3  True
1    Oslo      2.0        35.6   NaN
2  Bergen      4.5        40.1   NaN
```

The first assignment computed a Fahrenheit value for every row and added the column. The second used
`loc` to set `frost` only in the row below zero, so the column was created with `NaN` in the other
rows, and both lines changed `readings` itself.

## Setup

Three imports for the work, three for building the readings, two tables and a helper.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `numpy`, imported as `np`, provides `np.where` for a column chosen by a condition
- `warnings` catches the warnings pandas prints, so that `print_warnings` can show each one as its
  category and its first line
- `math`, `datetime` and `timedelta` build the readings

`readings` holds every hour of 2025 at the four stations, by the same formula as the **sqlite3, Deep
Dive** guide, with Svalbard's 24 silent hours on 2 March as `NaN`. `stations` holds each station's
latitude, and Kirkenes, which has no readings. pandas prints a warning with the name of the cell's
file and the line, which differ from machine to machine, so the cells that expect one catch it and
pass it to `print_warnings`.

In [1]:
import math
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})


def print_warnings(caught):
    """Print each caught warning as its category and the first line of its message."""
    for warning in caught:
        print(f"{warning.category.__name__}: {str(warning.message).strip().splitlines()[0]}")


print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


## Worked examples

### A column from arithmetic

The examples change `table`, a copy of `readings`, so that `readings` stays as Setup made it:

In [2]:
table = readings.copy()
table["fahrenheit"] = table["celsius"] * 9 / 5 + 32

print(readings.columns.tolist())
print(table.columns.tolist())
table.head(3)

['station', 'time', 'celsius']
['station', 'time', 'celsius', 'fahrenheit']


,station,time,celsius,fahrenheit
0,Bergen,2025-01-01,-3.6,25.52
1,Oslo,2025-01-01,-3.5,25.70
2,Svalbard,2025-01-01,-14.6,5.72


The right side was computed for all 35,040 rows at once and stored as a new column at the end of
`table`. `readings` kept its three columns, because `table` is a copy with data of its own.

### A column from a condition

A comparison gives `True` or `False` for every row, and that can be a column by itself. `np.where`
picks one of two values for each row from the same condition:

In [3]:
table["thaw"] = table["celsius"] >= 0
table["state"] = np.where(table["celsius"] >= 0, "thaw", "frost")

print(table["state"].value_counts())
print(table.loc[table["celsius"].isna(), "state"].unique())

state
thaw     22874
frost    12166
Name: count, dtype: int64
['frost']


Svalbard's missing hours came out as `frost`, because a comparison with `NaN` is false, so `np.where`
took the second value for them. The **Missing Data** notebook covers ways to keep a missing reading
missing.

### A column from another table

`map` looks each station's name up in a Series and returns the value it finds there, one for every
row:

In [4]:
latitude = stations.set_index("station")["latitude"]
table["latitude"] = table["station"].map(latitude)

table[["station", "celsius", "latitude"]].head(4)

,station,celsius,latitude
0,Bergen,-3.6,60.39
1,Oslo,-3.5,59.91
2,Svalbard,-14.6,78.22
3,Tromso,-6.7,69.65


`latitude` is indexed by station name, and `map` used those labels, so every row got its own
station's latitude. The **Merging** notebook joins whole tables in the same way.

### assign returns a new table

`assign` takes the new columns as keyword arguments. A function given as the value receives the table
and returns the column:

In [5]:
converted = readings.assign(
    fahrenheit=lambda d: d["celsius"] * 9 / 5 + 32,
    kelvin=lambda d: (d["celsius"] + 273.15).round(2),
)

print(readings.columns.tolist())
print(converted.columns.tolist())
print(-3.6 + 273.15)
converted.head(2)

['station', 'time', 'celsius']
['station', 'time', 'celsius', 'fahrenheit', 'kelvin']
269.54999999999995


,station,time,celsius,fahrenheit,kelvin
0,Bergen,2025-01-01,-3.6,25.52,269.55
1,Oslo,2025-01-01,-3.5,25.70,269.65


`readings` is unchanged and `converted` has two more columns. `round(2)` is there because adding
273.15 to -3.6 gives `269.54999999999995` in floating point, as the third line shows. Because `assign`
returns a table, another method can be called on its result in the same expression, as in
`readings.assign(...).groupby("station")`.

### Changing chosen rows with loc

Suppose the Tromso sensor read half a degree high for the first seven days of March. A mask picks those
rows, and `loc` with the mask and the column name changes them in `table`:

In [6]:
week = (table["station"] == "Tromso") & (table["time"] >= "2025-03-01") & (table["time"] < "2025-03-08")
before = table.loc[week, "celsius"].head(3).tolist()

table.loc[week, "celsius"] = (table.loc[week, "celsius"] - 0.5).round(1)

print(week.sum(), "readings corrected")
print(before, table.loc[week, "celsius"].head(3).tolist())

168 readings corrected
[-4.9, -5.1, -6.8] [-5.4, -5.6, -7.3]


All 168 readings, seven days of 24 hours, are half a degree lower. `table["time"] >= "2025-03-01"` compares dates and times with a date written as text, which
pandas parses for the comparison, and the **Boolean Filtering** notebook combines masks with `&`.

### rename and drop

Both return a new table. `rename` takes a dictionary from old name to new, and `drop` takes the names
to remove:

In [7]:
tidy = table.rename(columns={"celsius": "temp_c", "fahrenheit": "temp_f"}).drop(columns=["thaw", "latitude"])

tidy.columns.tolist()

['station', 'time', 'temp_c', 'temp_f', 'state']

### Changing a filtered table

Here is the warning this notebook promised. `warm` holds the readings above 15 degrees, and a column
is added to it:

In [8]:
warm = readings[readings["celsius"] > 15]

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    warm["hot"] = True

print_warnings(caught)
print("warm has hot:", "hot" in warm.columns, "| readings has hot:", "hot" in readings.columns)
print("warm shares memory with readings:", np.shares_memory(warm["celsius"].to_numpy(), readings["celsius"].to_numpy()))

warm has hot: True | readings has hot: False
warm shares memory with readings: False


The column went into `warm` and not into `readings`, which is what was wanted here, and `warm`'s
readings are its own, as `np.shares_memory` shows. pandas warned anyway, because `warm` was cut from
`readings` and pandas could not tell whether the change was meant to reach `readings` too. `.copy()`
makes `warm` a table of its own, and the warning goes away:

In [9]:
warm = readings[readings["celsius"] > 15].copy()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    warm["hot"] = True

print(len(caught), "warnings |", len(warm), "warm readings")

0 warnings | 2804 warm readings


### Copy-on-Write

A column taken out of a table can share memory with it. Change the column, and in pandas 2.2's
default mode the table changes too:

In [10]:
first = table["celsius"].iloc[0]
column = table["celsius"]
print("column shares memory with table:", np.shares_memory(column.to_numpy(), table["celsius"].to_numpy()))

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    column.iloc[0] = 99.0

print_warnings(caught)
print("table's first reading:", table["celsius"].iloc[0])

table.loc[0, "celsius"] = first                    # put the reading back

column shares memory with table: True
table's first reading: 99.0


Changing `column` changed `table`, with a warning. With Copy-on-Write switched on, the same lines leave
`table` alone. `pd.option_context` switches the option on for the indented block and back off at its
end:

In [11]:
with pd.option_context("mode.copy_on_write", True):
    column = table["celsius"]
    column.iloc[0] = 99.0
    print("table's first reading:", table["celsius"].iloc[0])

print("Copy-on-Write after the block:", pd.get_option("mode.copy_on_write"))

table's first reading: -3.6
Copy-on-Write after the block: False


Under Copy-on-Write, a table or column taken from another shares memory only until one of them is
changed, and the change then goes into a copy. A change can never leak back into the table it came
from, which is why that mode has no need for `SettingWithCopyWarning`.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/11-adding-and-changing-columns-solutions.ipynb).

**1.** Make `table` again with `readings.copy()`, add a column `kelvin` holding each reading plus 273.15, rounded to two places, and show the first three rows.

In [12]:
# your code here

**2.** Use `assign` to make a table with a column `month`, from `time` with `.dt.month`, and print how many readings fall in July.

In [13]:
# your code here

**3.** In a copy of `readings`, set `celsius` to `NaN` for Bergen's readings on 1 January with `loc`, and print how many of Bergen's readings are now missing.

In [14]:
# your code here

**4.** Make a table of Oslo's readings with `.copy()`, add a `fahrenheit` column to it inside `warnings.catch_warnings(record=True)`, and print how many warnings were caught.

In [15]:
# your code here

**5.** Make a new table with `celsius` renamed to `temp_c`, and check that `readings` still has a column called `celsius`.

In [16]:
# your code here

## Common errors

### SettingWithCopyWarning: A value is trying to be set on a copy of a slice from a DataFrame.

The aim is to cap every reading above 20 degrees at 20, in `readings` itself:

In [17]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    readings[readings["celsius"] > 20]["celsius"] = 20

print_warnings(caught)
print("warmest reading:", readings["celsius"].max())

warmest reading: 20.8


This is chained assignment. `readings[readings["celsius"] > 20]` made a new table, `["celsius"] = 20`
changed that new table, and the new table was thrown away, so the warmest reading is still 20.8. Do it
in one step with `loc`, shown here on a copy so that `readings` stays as it was:

In [18]:
capped = readings.copy()
capped.loc[capped["celsius"] > 20, "celsius"] = 20

print("warmest reading:", capped["celsius"].max())

warmest reading: 20.0


### ChainedAssignmentError: A value is trying to be set on a copy of a DataFrame or Series through chained assignment.

In [19]:
with pd.option_context("mode.copy_on_write", True):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        readings[readings["celsius"] > 20]["celsius"] = 20

print_warnings(caught)
print("warmest reading:", readings["celsius"].max())

ChainedAssignmentError: A value is trying to be set on a copy of a DataFrame or Series through chained assignment.
warmest reading: 20.8


The same line under Copy-on-Write gets a different warning and the same result. In that mode the
intermediate table always behaves as a copy, so chained assignment can never change `readings`.
`ChainedAssignmentError` is a warning despite its name, and the cell went on to
print the reading. The fix is the same `loc` in one step.

### ValueError: Length of values (3) does not match length of index (35040)

In [20]:
table["checked"] = [True, False, True]

ValueError: Length of values (3) does not match length of index (35040)

A list put into a column needs one value for each of the 35,040 rows. A single value, as in
`table["checked"] = False`, goes into every row.

### No error, and a column of NaN

The aim is each station's mean beside every reading:

In [21]:
station_means = readings.groupby("station")["celsius"].mean()
table["station_mean"] = station_means

print(table["station_mean"].isna().sum(), "of", len(table), "values are NaN")

35040 of 35040 values are NaN


`station_means` is indexed by station name, and `table` by row number, from 0 to 35039. Assignment
lined the Series up by label, found no label the two share, and stored `NaN` in every row. Look each
station up with `map` instead, or use `transform`, which the **Aggregating** notebook covers:

In [22]:
table["station_mean"] = table["station"].map(station_means).round(2)

table[["station", "celsius", "station_mean"]].head(4)

,station,celsius,station_mean
0,Bergen,-3.6,8.00
1,Oslo,-3.5,6.50
2,Svalbard,-14.6,-4.48
3,Tromso,-6.7,3.50


### No error, and the column keeps its old name

In [23]:
renamed = readings.rename(columns={"Celsius": "temp_c"})

print(renamed.columns.tolist())

['station', 'time', 'celsius']


`rename` skips any name in the dictionary that the table does not have, and `Celsius` is not
`celsius`, so nothing changed. Passing `errors="raise"` makes a name the table does not have raise a
`KeyError` instead:

In [24]:
try:
    readings.rename(columns={"Celsius": "temp_c"}, errors="raise")
except KeyError as error:
    print("KeyError:", error)

KeyError: "['Celsius'] not found in axis"


## Recap

- Add a column with `df["name"] = ...`, from arithmetic, a condition, a single value or `map`
- Use `assign` for a new table with extra columns, leaving the original as it was
- Change chosen rows of one column with `df.loc[mask, "name"] = value`, in a single step
- Read `SettingWithCopyWarning` as a change that may not have reached the original, and fix it with
  `.copy()` or with `loc`
- Switch Copy-on-Write on with `pd.option_context` to see how pandas 3 treats a table taken from another

## What is next

The **groupby** notebook splits the table into groups, such as one for each station, and shows what
pandas holds between the split and the answer.

---

&#8592; **Previous:** [Missing Data](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/10-missing-data.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [groupby](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/12-groupby.ipynb) &#8594;
